In [ ]:
!pip install -q peft transformers diffusers accelerate datasets scipy

import os
import torch
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms
from scipy.stats import ortho_group
from datasets import load_dataset
from diffusers import AutoencoderKL, UNet2DConditionModel, DDPMScheduler
from transformers import CLIPTextModel, CLIPTextModelWithProjection, CLIPTokenizer
from peft import LoraConfig, get_peft_model
from tqdm.auto import tqdm

In [ ]:
MODEL_ID = "stabilityai/stable-diffusion-xl-base-1.0"
DATASET_ID = "lambdalabs/pokemon-blip-captions"
LATENT_DIR = "/kaggle/working/cached_latents"
OUTPUT_DIR = "/kaggle/working/crypto_lora_output"

os.makedirs(LATENT_DIR, exist_ok=True)
os.makedirs(OUTPUT_DIR, exist_ok=True)

key_matrix_np = ortho_group.rvs(dim=4, random_state=42)
cipher_key = torch.tensor(key_matrix_np, dtype=torch.float32, device="cuda")
torch.save(cipher_key, os.path.join(OUTPUT_DIR, "latent_cipher_key.pt"))

In [ ]:
vae = AutoencoderKL.from_pretrained(MODEL_ID, subfolder="vae", torch_dtype=torch.float16).to("cuda")
tokenizer_1 = CLIPTokenizer.from_pretrained(MODEL_ID, subfolder="tokenizer")
tokenizer_2 = CLIPTokenizer.from_pretrained(MODEL_ID, subfolder="tokenizer_2")
text_encoder_1 = CLIPTextModel.from_pretrained(MODEL_ID, subfolder="text_encoder", torch_dtype=torch.float16).to("cuda")
text_encoder_2 = CLIPTextModelWithProjection.from_pretrained(MODEL_ID, subfolder="text_encoder_2", torch_dtype=torch.float16).to("cuda")

vae.eval()
text_encoder_1.eval()
text_encoder_2.eval()

image_transforms = transforms.Compose([
    transforms.Resize(1024, interpolation=transforms.InterpolationMode.BILINEAR),
    transforms.CenterCrop(1024),
    transforms.ToTensor(),
    transforms.Normalize([0.5], [0.5]),
])

dataset = load_dataset(DATASET_ID, split="train")

for idx, item in tqdm(enumerate(dataset), total=min(len(dataset), 1000), desc="Caching latents"):
    if idx >= 1000:
        break
    try:
        image = item["image"].convert("RGB")
        pixel_values = image_transforms(image).unsqueeze(0).to("cuda", dtype=torch.float16)
        
        with torch.no_grad():
            latents = vae.encode(pixel_values).latent_dist.sample()
            latents = latents * vae.config.scaling_factor
            
            latents_enc = torch.einsum('ij, b j h w -> b i h w', cipher_key.to(torch.float16), latents)
            
            tok_1 = tokenizer_1(item["text"], padding="max_length", max_length=77, truncation=True, return_tensors="pt").input_ids.to("cuda")
            tok_2 = tokenizer_2(item["text"], padding="max_length", max_length=77, truncation=True, return_tensors="pt").input_ids.to("cuda")
            
            out_1 = text_encoder_1(tok_1, output_hidden_states=True)
            out_2 = text_encoder_2(tok_2, output_hidden_states=True)
            
            prompt_embeds = torch.cat([out_1.hidden_states[-2], out_2.hidden_states[-2]], dim=-1)
            text_embeds = out_2.text_embeds
            
        torch.save({
            "latent": latents_enc.squeeze(0).cpu(),
            "prompt_embeds": prompt_embeds.squeeze(0).cpu(),
            "text_embeds": text_embeds.squeeze(0).cpu()
        }, os.path.join(LATENT_DIR, f"sample_{idx}.pt"))
        
    except Exception:
        continue

del vae, text_encoder_1, text_encoder_2, tokenizer_1, tokenizer_2
torch.cuda.empty_cache()

In [ ]:
class CachedDataset(Dataset):
    def __init__(self, latent_dir):
        self.files = [os.path.join(latent_dir, f) for f in os.listdir(latent_dir) if f.endswith(".pt")]
    def __len__(self):
        return len(self.files)
    def __getitem__(self, idx):
        return torch.load(self.files[idx], weights_only=True)

train_dataset = CachedDataset(LATENT_DIR)
train_dataloader = DataLoader(train_dataset, batch_size=2, shuffle=True, num_workers=2)

unet = UNet2DConditionModel.from_pretrained(MODEL_ID, subfolder="unet", torch_dtype=torch.float16).to("cuda")
unet.requires_grad_(False)
unet.enable_gradient_checkpointing()

lora_config = LoraConfig(
    r=32,
    lora_alpha=64,
    init_lora_weights="gaussian",
    target_modules=["to_k", "to_q", "to_v", "to_out.0", "add_k_proj", "add_v_proj"]
)

unet = get_peft_model(unet, lora_config)

for param in unet.parameters():
    if param.requires_grad:
        param.data = param.data.to(torch.float32)

In [ ]:
trainable_params = [p for p in unet.parameters() if p.requires_grad]
print(f"Trainable params: {sum(p.numel() for p in trainable_params):,}")
optimizer = torch.optim.AdamW(trainable_params, lr=1e-4)
noise_scheduler = DDPMScheduler.from_pretrained(MODEL_ID, subfolder="scheduler")
scaler = torch.amp.GradScaler('cuda')

add_time_ids = torch.tensor([[1024, 1024, 0, 0, 1024, 1024]], dtype=torch.float32).to("cuda")

unet.train()
EPOCHS = 10
total_steps = EPOCHS * len(train_dataloader)
lr_scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=total_steps)

for epoch in range(EPOCHS):
    epoch_loss = 0
    for step, batch in enumerate(train_dataloader):
        latents = batch["latent"].to("cuda", dtype=torch.float32)
        prompt_embeds = batch["prompt_embeds"].to("cuda", dtype=torch.float32)
        text_embeds = batch["text_embeds"].to("cuda", dtype=torch.float32)
        
        noise = torch.randn_like(latents)
        bsz = latents.shape[0]
        timesteps = torch.randint(0, noise_scheduler.config.num_train_timesteps, (bsz,), device="cuda").long()
        
        noisy_latents = noise_scheduler.add_noise(latents, noise, timesteps)
        
        added_cond_kwargs = {
            "text_embeds": text_embeds,
            "time_ids": add_time_ids.repeat(bsz, 1)
        }
        
        with torch.amp.autocast('cuda', dtype=torch.float16):
            noise_pred = unet(
                noisy_latents.to(torch.float16),
                timesteps,
                encoder_hidden_states=prompt_embeds.to(torch.float16),
                added_cond_kwargs=added_cond_kwargs
            ).sample
            
            loss = F.mse_loss(noise_pred.float(), noise.float(), reduction="mean")
            
        scaler.scale(loss).backward()
        scaler.unscale_(optimizer)
        torch.nn.utils.clip_grad_norm_(trainable_params, max_norm=1.0)
        scaler.step(optimizer)
        scaler.update()
        lr_scheduler.step()
        optimizer.zero_grad(set_to_none=True)
        
        epoch_loss += loss.item()
        
        if step % 50 == 0:
            print(f"Epoch {epoch+1} | Step {step} | Loss: {loss.item():.4f} | LR: {lr_scheduler.get_last_lr()[0]:.2e}")

    # ── Save checkpoint every epoch ──────────────────────────────────────
    epoch_ckpt = os.path.join(OUTPUT_DIR, f"epoch_{epoch + 1}")
    unet.save_pretrained(epoch_ckpt)
    print(f"  ✓ Epoch {epoch+1} checkpoint → {epoch_ckpt}  (avg loss: {epoch_loss / len(train_dataloader):.4f})")

unet.save_pretrained(OUTPUT_DIR)
print("Training complete. Final weights saved.")